GPU Check

In [ ]:
import torch
print("GPU Available:", torch.cuda.is_available())
print("GPU Name:", torch.cuda.get_device_name(0))

Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Install Dependencies

In [ ]:
!pip install ultralytics matplotlib seaborn opencv-python

Load Dataset Path

In [ ]:
dataset_path = "/content/drive/MyDrive/Traffic-signs dataset"

Fix data.yaml

In [ ]:
import yaml

yaml_path = dataset_path + "/data.yaml"

with open(yaml_path, 'r') as f:
    data = yaml.safe_load(f)

data['path'] = dataset_path

with open(yaml_path, 'w') as f:
    yaml.dump(data, f)

print("Updated YAML:", data)

Train YOLOv8 (GPU Optimized)

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

model.train(
    data=dataset_path + "/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    device=0   # GPU
)

Visualize Training Results

In [ ]:
from IPython.display import Image, display

display(Image(filename='runs/detect/train/results.png'))

Confusion Matrix

In [ ]:
display(Image(filename='runs/detect/train/confusion_matrix.png'))

Precision-Recall Curve

In [ ]:
display(Image(filename='/content/runs/detect/train/BoxPR_curve.png'))

F1 Score Curve

In [ ]:
display(Image(filename='/content/runs/detect/train/BoxF1_curve.png'))

Evaluate Model

In [ ]:
metrics = model.val()

print(metrics)

Test Predictions (Visualization)


In [ ]:
model = YOLO("runs/detect/train/weights/best.pt")

results = model(dataset_path + "/test/images", save=True)

Show Predictions

In [ ]:
import cv2
import matplotlib.pyplot as plt
import os

pred_folder = "runs/detect/predict"

images = os.listdir(pred_folder)

for i in range(3):  # show 3 images
    img_path = pred_folder + "/" + images[i]
    img = cv2.imread(img_path)
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title("Prediction")
    plt.axis("off")
    plt.show()

Custom Visualization (Bar Chart of Classes)

In [ ]:
import os

label_path = dataset_path + "/train/labels"

class_count = {}

for file in os.listdir(label_path):
    with open(os.path.join(label_path, file)) as f:
        for line in f:
            cls = int(line.split()[0])
            class_count[cls] = class_count.get(cls, 0) + 1

# Plot
import matplotlib.pyplot as plt

plt.bar(class_count.keys(), class_count.values())
plt.xlabel("Class ID")
plt.ylabel("Count")
plt.title("Dataset Distribution")
plt.show()

Detection Function

In [ ]:
def detect_sign(image_path):
    results = model(image_path)
    results[0].show()
    return results[0].boxes.cls.tolist()

Run Detection

In [ ]:
image_path = "/content/drive/MyDrive/Traffic-signs dataset/test/images/BA-11-_png_jpg.rf.0c99f050c078387e61fb32d10c2e63fc.jpg"

labels = detect_sign(image_path)
print("Detected Classes:", labels)

#**Integrated YOLO and Reinforcement Learning**

In [ ]:
!pip install ultralytics opencv-python gym numpy matplotlib

In [ ]:
import os
import cv2
import gym
import random
import numpy as np
import matplotlib.pyplot as plt

from ultralytics import YOLO

In [ ]:
!unzip imgs.zip

In [ ]:
model = YOLO("yolov8n.pt")

Vehicle Detection Function

In [ ]:
vehicle_classes = ['car', 'bus', 'truck', 'motorcycle']

def count_vehicles_by_road(image_path, show=False):

    img = cv2.imread(image_path)

    h, w, _ = img.shape

    # Road Regions
    roads = {
        "North": 0,
        "South": 0,
        "East": 0,
        "West": 0
    }

    results = model(img)

    for r in results:

        for box in r.boxes:

            cls = int(box.cls[0])

            label = model.names[cls]

            if label in vehicle_classes:

                x1, y1, x2, y2 = map(int, box.xyxy[0])

                cx = (x1 + x2) // 2
                cy = (y1 + y2) // 2

                # Assign to road
                if cy < h // 2:
                    roads["North"] += 1
                else:
                    roads["South"] += 1

                if cx < w // 2:
                    roads["West"] += 1
                else:
                    roads["East"] += 1

                # Draw Box
                cv2.rectangle(img, (x1, y1), (x2, y2),
                              (0, 255, 0), 2)

                cv2.putText(img,
                            label,
                            (x1, y1 - 5),
                            cv2.FONT_HERSHEY_SIMPLEX,
                            0.5,
                            (255, 0, 0),
                            2)

    if show:

        print("Vehicle Count Per Road:")
        print(roads)

        plt.figure(figsize=(10, 8))

        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

        plt.axis("off")

        plt.show()

    return roads

Test YOLO Detection

In [ ]:
test_image = "imgs/1.jpg"

counts = count_vehicles_by_road(test_image, show=True)

print(counts)

RL Traffic Environment

In [ ]:
class TrafficEnv:

    def __init__(self):

        self.roads = ["North", "South", "East", "West"]

        self.queues = {
            road: 0 for road in self.roads
        }

        self.time_step = 0

    def reset(self):

        self.queues = {
            road: 0 for road in self.roads
        }

        self.time_step = 0

        return np.array(list(self.queues.values()))

    def step(self, action, detected_counts):

        selected_road = self.roads[action]

        # Increase queues from detected traffic
        for road in self.roads:

            incoming = detected_counts[road]

            # Simulate new incoming traffic
            incoming += random.randint(0, 3)

            self.queues[road] += incoming

        # Green light decreases traffic
        passed = min(8, self.queues[selected_road])

        self.queues[selected_road] -= passed

        # Avoid negative queues
        for road in self.roads:

            self.queues[road] = max(0, self.queues[road])

        self.time_step += 1

        next_state = np.array(list(self.queues.values()))

        # Reward for reducing congestion
        reward = -sum(self.queues.values())

        done = self.time_step >= 20

        return next_state, reward, done

Q Learning Setup

In [ ]:
Q = np.zeros((100, 4))

alpha = 0.1
gamma = 0.9
epsilon = 0.2

Load Image Paths

In [ ]:
image_folder = "imgs"

image_list = []

for img in os.listdir(image_folder):

    if img.endswith(".jpg"):

        image_list.append(
            os.path.join(image_folder, img)
        )

print("Total Images:", len(image_list))

RL Training

#EPISODES = 5 as Github has restrictions


In [ ]:
env = TrafficEnv()

episodes = 5

reward_history = []

for ep in range(episodes):

    state = env.reset()

    total_reward = 0

    print(f"\n=========== Episode {ep+1} ===========")

    for step in range(10):

        # Sequential images
        image_path = image_list[step % len(image_list)]

        # Vehicle detection
        counts = count_vehicles_by_road(
            image_path,
            show=True
        )

        total_vehicles = sum(counts.values())

        current_state = min(total_vehicles, 99)

        # Epsilon Greedy
        if random.uniform(0, 1) < epsilon:

            action = random.randint(0, 3)

        else:

            action = np.argmax(Q[current_state])

        next_state, reward, done = env.step(
            action,
            counts
        )

        next_total = min(sum(next_state), 99)

        # Q Update
        Q[current_state, action] = (
            Q[current_state, action]
            + alpha * (
                reward
                + gamma * np.max(Q[next_total])
                - Q[current_state, action]
            )
        )

        total_reward += reward

        roads = ["North", "South", "East", "West"]

        print(f"\nStep {step+1}")

        print(f"Image: {image_path}")

        print(f"Detected Vehicles: {counts}")

        print(f"GREEN Light Given To: {roads[action]}")

        print(f"Current Queues: {env.queues}")

        print(f"Reward: {reward}")

        if done:
            break

    reward_history.append(total_reward)

Training Graph

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(reward_history, marker='o')

plt.xlabel("Episodes")

plt.ylabel("Total Reward")

plt.title("RL Training Performance")

plt.grid(True)

plt.show()

Final Demo

In [ ]:
"""Dynamic Traffic Simulation Demo"""

from IPython.display import clear_output
import time

demo_env = TrafficEnv()

roads = ["North", "South", "East", "West"]

simulation_steps = 20

for step in range(simulation_steps):

    # Select image dynamically
    image_path = image_list[step % len(image_list)]

    # Detect vehicles
    counts = count_vehicles_by_road(
        image_path,
        show=False
    )

    # Add dynamic incoming traffic
    for road in roads:
        counts[road] += random.randint(0, 3)

    total_vehicles = sum(counts.values())

    current_state = min(total_vehicles, 99)

    # RL chooses best action
    action = np.argmax(list(counts.values()))

    # Update environment
    next_state, reward, done = demo_env.step(
        action,
        counts
    )

    green_road = roads[action]

    # Clear previous output
    clear_output(wait=True)

    # Visualization
    plt.figure(figsize=(8,5))

    queue_values = list(demo_env.queues.values())

    colors = []

    for road in roads:

        if road == green_road:
            colors.append("green")

        else:
            colors.append("red")

    plt.bar(
        roads,
        queue_values,
        color=colors
    )

    plt.title(f"Smart Traffic Signal Simulation - Step {step+1}")

    plt.xlabel("Roads")

    plt.ylabel("Vehicles Waiting")

    # Queue values on top
    for i, v in enumerate(queue_values):

        plt.text(i, v + 0.3, str(v), ha='center')

    plt.show()

    # Console Status
    print("\n========= LIVE TRAFFIC STATUS =========")

    print(f"Step: {step+1}")

    print(f"\nGREEN SIGNAL --> {green_road}")

    print("\nCurrent Traffic Queues:")

    for road in roads:

        signal = "GREEN" if road == green_road else "RED"

        print(
            f"{road}: {signal} | Waiting Vehicles: {demo_env.queues[road]}"
        )

    print(f"\nReward: {reward}")

    time.sleep(1)

print("\n========= PERFORMANCE SUMMARY =========")

total_waiting = sum(demo_env.queues.values())

average_wait = total_waiting / 4

print(f"Total Waiting Vehicles: {total_waiting}")

print(f"Average Waiting Vehicles: {average_wait:.2f}")